# Generators and Itertools

Generators are one of Python's most elegant features. A generator function produces values one at a time with `yield`, consuming only the memory needed for the current value, not the entire sequence. `itertools` extends this with a library of composable building blocks: infinite sequences, combinatorics, and grouping operations that would take dozens of lines elsewhere.

**What's inside:** generator functions, `yield`, `yield from`, generator expressions, and `itertools`: `chain`, `islice`, `cycle`, `accumulate`, `combinations`, `permutations`, `product`, `groupby`.

**Learn more:** [Generator expressions](https://docs.python.org/3/glossary.html#term-generator) · [itertools](https://docs.python.org/3/library/itertools.html)

## 1. Generator Functions

A function with `yield` becomes a generator; it pauses at each `yield` and resumes where it left off when `next()` is called.

### 1.1 yield

In [2]:
def countdown(n):
    while n > 0:
        yield n
        n -= 1

for x in countdown(5):
    print(x, end=' ')

5 4 3 2 1 

In [3]:
# a generator is an iterator (call next() manually)
g = countdown(3)
print(next(g))
print(next(g))
print(next(g))

3
2
1


### 1.2 Lazy evaluation: only compute what you need

In [4]:
import sys

# a list holds all values in memory at once
list_version = [x**2 for x in range(100_000)]

# a generator computes each value on demand
def squares(n):
    for x in range(n):
        yield x**2

gen_version = squares(100_000)

print(f'list size: {sys.getsizeof(list_version):,} bytes')
print(f'gen size:  {sys.getsizeof(gen_version):,} bytes')

list size: 800,984 bytes
gen size:  208 bytes


In [5]:
# infinite generator (impossible with a list)
def naturals(start=1):
    n = start
    while True:
        yield n
        n += 1

# take only what you need
from itertools import islice
list(islice(naturals(), 10))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

### 1.3 yield from: delegate to another iterable

In [6]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # recurse into sub-lists
        else:
            yield item

list(flatten([1, [2, [3, 4]], [5, 6]]))

[1, 2, 3, 4, 5, 6]

## 2. Generator Expressions

Like list comprehensions but with parentheses; lazy by default, no list is built.

In [7]:
# list comprehension (builds the entire list)
squares_list = [x**2 for x in range(10)]

# generator expression (lazy)
squares_gen = (x**2 for x in range(10))

print(type(squares_gen))
print(list(squares_gen))

<class 'generator'>
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


In [8]:
# pass a generator expression directly to a function (no extra parens needed)
result = sum(x**2 for x in range(1000))
result

332833500

In [9]:
# short-circuit: stop as soon as the condition is met
first_over_100 = next(x**2 for x in range(1000) if x**2 > 100)
first_over_100

121

## 3. Infinite Iterators (itertools)

In [10]:
from itertools import count, cycle, repeat, islice

# count: start, [step]
list(islice(count(10, 5), 6))   # 10, 15, 20, ...

[10, 15, 20, 25, 30, 35]

In [11]:
# cycle: loop a sequence forever
list(islice(cycle('ABC'), 8))

['A', 'B', 'C', 'A', 'B', 'C', 'A', 'B']

In [12]:
# repeat: a value n times (or forever)
list(repeat('hello', 4))

['hello', 'hello', 'hello', 'hello']

## 4. Chaining and Slicing Iterables

In [13]:
from itertools import chain

# chain: iterate over multiple iterables as if they were one
list(chain([1, 2], [3, 4], [5, 6]))

[1, 2, 3, 4, 5, 6]

In [14]:
# chain.from_iterable: flatten one level of nesting
nested = [[1, 2], [3, 4], [5, 6]]
list(chain.from_iterable(nested))

[1, 2, 3, 4, 5, 6]

In [15]:
# islice: slice any iterator (start, stop, step) without building a list
list(islice(range(100), 10, 20, 2))

[10, 12, 14, 16, 18]

## 5. Combinatorics

In [16]:
from itertools import combinations, permutations, product

# combinations: choose r items from a sequence (order doesn't matter)
list(combinations('ABCD', 2))

[('A', 'B'), ('A', 'C'), ('A', 'D'), ('B', 'C'), ('B', 'D'), ('C', 'D')]

In [17]:
# permutations: ordered arrangements
list(permutations('ABC', 2))

[('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]

In [18]:
# product: cartesian product (like nested for loops)
list(product([1, 2], ['a', 'b', 'c']))

[(1, 'a'), (1, 'b'), (1, 'c'), (2, 'a'), (2, 'b'), (2, 'c')]

In [19]:
# repeat= repeats the iterable with itself (like a nested loop over the same sequence)
list(product(range(2), repeat=3))   # all 3-bit binary numbers

[(0, 0, 0),
 (0, 0, 1),
 (0, 1, 0),
 (0, 1, 1),
 (1, 0, 0),
 (1, 0, 1),
 (1, 1, 0),
 (1, 1, 1)]

## 6. Aggregating and Grouping

### 6.1 accumulate: running totals

In [20]:
from itertools import accumulate
import operator

sales = [100, 200, 150, 300, 250]

list(accumulate(sales))                          # running sum
list(accumulate(sales, operator.mul))            # running product
list(accumulate(sales, max))                     # running maximum

[100, 200, 200, 300, 300]

### 6.2 groupby: group consecutive items by a key

Input must be sorted by the key first; `groupby` only groups *consecutive* equal-key items.

In [21]:
from itertools import groupby

people = [
    {'name': 'Alice', 'city': 'Austin'},
    {'name': 'Carol', 'city': 'Austin'},
    {'name': 'Bob',   'city': 'Boston'},
    {'name': 'Eve',   'city': 'Boston'},
    {'name': 'Dave',  'city': 'Denver'},
]

# sort by key first, then group
people_sorted = sorted(people, key=lambda p: p['city'])

for city, group in groupby(people_sorted, key=lambda p: p['city']):
    names = [p['name'] for p in group]
    print(f'{city}: {names}')

Austin: ['Alice', 'Carol']
Boston: ['Bob', 'Eve']
Denver: ['Dave']


### 6.3 compress and filterfalse: selective filtering

In [22]:
from itertools import compress, filterfalse

data     = ['a', 'b', 'c', 'd', 'e']
selector = [1, 0, 1, 0, 1]

list(compress(data, selector))              # keep items where selector is truthy

['a', 'c', 'e']

In [23]:
# filterfalse: opposite of filter (keep items where predicate is False)
list(filterfalse(lambda x: x % 2 == 0, range(10)))

[1, 3, 5, 7, 9]